# ML-10 — Content Action Playbook

The original assignment uses the FlyRank Hugging Face warehouse dataset. Due to repeated access issues with the gated dataset, I completed this notebook using the dataset provided inside the internship repository:

`../../data/raw/content_refresh_anonymized.csv`

This CSV contains the same type of anonymized search performance data required for learning the concepts of data contracts, feature engineering, and data leakage. All queries, feature engineering, and verification steps in this notebook are therefore performed on the repository CSV instead of the Hugging Face warehouse tables.

## 1. Ranked actions + reason codes

The ranked queue turns the baseline signals into practical content actions.

The main actions are:

- Refresh content when it is old and has signs of declining performance.
- Improve CTR when impressions are present but click-through performance is weak.
- Prioritize quick-win opportunities when search volume is relatively high.
- Monitor content when the signals are not strong enough for a direct action.

Each row receives one reason code so that a content team can understand why the item was ranked. The queue is intended for decision-support and human review, not automatic publishing or deletion.

In [1]:
import os
import pandas as pd
import numpy as np

# Load the baseline ranked queue if it already exists

baseline_path = "../../work/outputs/baseline_action_score.csv"
raw_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(baseline_path):
    queue = pd.read_csv(baseline_path)
    print("Loaded baseline ranked queue:")
    print(baseline_path)

else:
    # Fallback: create the queue directly from the repository CSV
    df = pd.read_csv(raw_path)

    # Make sure numeric columns are numeric
    numeric_columns = [
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update"
    ]

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Fill missing values
    for col in numeric_columns:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Create simple baseline score
    df["baseline_score"] = 0.0

    # Older content gets a higher refresh score
    df["baseline_score"] += (
        df["content_age_days"] >= df["content_age_days"].median()
    ).astype(int) * 2

    # Content with impressions but relatively weak CTR
    ctr_threshold = df["ctr"].median()

    df["baseline_score"] += (
        (df["impressions_90d"] > 0) &
        (df["ctr"] < ctr_threshold)
    ).astype(int) * 2

    # Higher search volume gets priority
    volume_threshold = df["search_volume"].median()

    df["baseline_score"] += (
        df["search_volume"] >= volume_threshold
    ).astype(int)

    # Reason code
    def get_reason(row):
        if (
            row["content_age_days"] >= df["content_age_days"].median()
            and row["impressions_90d"] > 0
        ):
            return "REFRESH_STALE"

        if (
            row["impressions_90d"] > 0
            and row["ctr"] < ctr_threshold
        ):
            return "IMPROVE_CTR"

        if row["search_volume"] >= volume_threshold:
            return "HIGH_VOLUME"

        return "MONITOR"

    df["reason_code"] = df.apply(get_reason, axis=1)

    # Action label
    action_map = {
        "REFRESH_STALE": "Refresh content",
        "IMPROVE_CTR": "Improve CTR",
        "HIGH_VOLUME": "Prioritize opportunity",
        "MONITOR": "Monitor"
    }

    df["action"] = df["reason_code"].map(action_map)

    queue = df.sort_values(
        "baseline_score",
        ascending=False
    ).reset_index(drop=True)

# Make sure expected fields exist

if "baseline_score" not in queue.columns:
    if "score" in queue.columns:
        queue["baseline_score"] = queue["score"]
    else:
        queue["baseline_score"] = 0

if "reason_code" not in queue.columns:
    queue["reason_code"] = "MONITOR"

if "action" not in queue.columns:
    queue["action"] = "Monitor"

# Add rank
queue["rank"] = range(1, len(queue) + 1)

# Display the ranked actions
display(
    queue[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

,rank,content_id,baseline_score,reason_code,action
0,1,content_ba8e51f13800,5.0,REFRESH_STALE,Refresh content
1,2,content_249298388b45,5.0,REFRESH_STALE,Refresh content
2,3,content_4be930227848,5.0,REFRESH_STALE,Refresh content
3,4,content_2dfd17269502,5.0,REFRESH_STALE,Refresh content
4,5,content_fe5d259e6bc5,5.0,REFRESH_STALE,Refresh content
5,6,content_cd850fb019b1,5.0,REFRESH_STALE,Refresh content
6,7,content_a00c249b224d,5.0,REFRESH_STALE,Refresh content
7,8,content_fd6261b74d30,5.0,REFRESH_STALE,Refresh content
8,9,content_d87a116e2c79,5.0,REFRESH_STALE,Refresh content
9,10,content_4595e8704e07,5.0,REFRESH_STALE,Refresh content


## 2. Intended use and limits

This playbook is intended to help a content team prioritize pages for review. It provides a ranked starting point using observable content and performance signals from the repository CSV.

The output is decision-support rather than an automatic decision. A high score does not mean that a page must be changed. Human reviewers should check the actual content, search intent, business context, and recent changes before taking action.

The playbook should not be used to automatically publish, delete, redirect, or rewrite content. Its recommendations are directional and depend on the quality and time period of the available data.

In [2]:
# Show the available signals used for decision support

signals = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

available_signals = [
    col for col in signals
    if col in queue.columns
]

print("Signals available for decision-support:")
for signal in available_signals:
    print("-", signal)

print("\nNumber of ranked content items:", len(queue))

print("\nAction counts:")
display(
    queue["action"].value_counts().rename_axis("action").reset_index(name="count")
)

Signals available for decision-support:
- search_volume
- impressions_90d
- clicks_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update

Number of ranked content items: 30000

Action counts:


,action,count
0,Refresh content,15210
1,Improve CTR,7174
2,Monitor,4358
3,Prioritize opportunity,3258


## 3. Human review + the no-go list

Every recommendation should be reviewed by a person before action is taken.

The reviewer should check whether the recommendation makes sense for the actual page, search intent, content quality, and current business context.

The following actions should not be automated from this baseline:

- Automatically publishing content changes.
- Automatically deleting pages.
- Automatically changing URLs or redirects.
- Automatically changing search intent classification.
- Automatically making decisions about important or sensitive content.
- Treating a high score as proof that a page is underperforming.

The model and rule output should only be used to decide which pages deserve human attention first.

In [3]:
# Human-review checklist for the highest-ranked items

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "baseline_score"
]

review_columns = [
    col for col in review_columns
    if col in queue.columns
]

top_review = queue.head(20)[review_columns].copy()

top_review["human_review_required"] = "Yes"

top_review["review_checks"] = (
    "Check content, search intent, recent changes, and business context"
)

display(top_review)

,rank,content_id,action,reason_code,baseline_score,human_review_required,review_checks
0,1,content_ba8e51f13800,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
1,2,content_249298388b45,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
2,3,content_4be930227848,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
3,4,content_2dfd17269502,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
4,5,content_fe5d259e6bc5,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
5,6,content_cd850fb019b1,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
6,7,content_a00c249b224d,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
7,8,content_fd6261b74d30,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
8,9,content_d87a116e2c79,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."
9,10,content_4595e8704e07,Refresh content,REFRESH_STALE,5.0,Yes,"Check content, search intent, recent changes, ..."


## 4. Monitoring / retrain triggers

The recommendations can become stale when the underlying content or search performance changes.

The queue should be reviewed again when the available performance data is updated, when content changes significantly, or when the relationship between the signals and useful actions changes.

A future model should be reconsidered if its measured performance becomes weaker than the baseline, if the error pattern changes, or if the available features no longer represent the decision being supported.

These triggers are monitoring guidelines rather than production automation rules.

In [4]:
# -Basic monitoring information
monitor_columns = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

monitor_columns = [
    col for col in monitor_columns
    if col in queue.columns
]

print("Monitoring signals:")

for col in monitor_columns:
    print(f"\n{col}")

    values = pd.to_numeric(
        queue[col],
        errors="coerce"
    )

    print("  Missing:", values.isna().sum())
    print("  Median:", round(values.median(), 4))
    print("  Minimum:", round(values.min(), 4))
    print("  Maximum:", round(values.max(), 4))

print("\nSuggested review triggers:")
print("1. New performance data becomes available.")
print("2. Content is substantially updated.")
print("3. Signal distributions change noticeably.")
print("4. Baseline recommendations become inconsistent with human review.")
print("5. A future model performs worse than the baseline.")

Monitoring signals:

search_volume
  Missing: 0
  Median: 10.0
  Minimum: 0.0
  Maximum: 74000.0

impressions_90d
  Missing: 0
  Median: 731.0
  Minimum: 1
  Maximum: 517715

clicks_90d
  Missing: 0
  Median: 1.0
  Minimum: 0
  Maximum: 4178

ctr
  Missing: 0
  Median: 0.07
  Minimum: 0.0
  Maximum: 100.0

avg_position
  Missing: 0
  Median: 10.8
  Minimum: 0.0
  Maximum: 245.0

content_age_days
  Missing: 0
  Median: 236.0
  Minimum: 90
  Maximum: 564

days_since_last_update
  Missing: 0
  Median: 20.0
  Minimum: 1
  Maximum: 373

Suggested review triggers:
1. New performance data becomes available.
2. Content is substantially updated.
3. Signal distributions change noticeably.
4. Baseline recommendations become inconsistent with human review.
5. A future model performs worse than the baseline.


## 5. Exports for the paper

The ranked queue is exported to `work/outputs/` so that the results can be reused in the research paper.

The CSV contains the ranking, score, reason code, and action label. The export is regenerated from the notebook rather than being treated as a manually maintained dataset.

The output is intended to support the recommendations section of the paper and should be interpreted as a decision-support queue rather than a production system.

In [ ]:
# Export ranked queue for the paper

output_dir = "../../work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "content_action_playbook.csv"
)

# Columns useful for the paper/playbook
export_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]

export_columns = [
    col for col in export_columns
    if col in queue.columns
]

paper_queue = queue[export_columns].copy()

paper_queue.to_csv(
    output_path,
    index=False
)

print("Export completed:")
print(output_path)

print("\nRows exported:", len(paper_queue))


display(
    paper_queue.head(20)
)

Export completed:
../../work/outputs/content_action_playbook.csv

Rows exported: 30000


,rank,content_id,baseline_score,reason_code,action
0,1,content_ba8e51f13800,5.0,REFRESH_STALE,Refresh content
1,2,content_249298388b45,5.0,REFRESH_STALE,Refresh content
2,3,content_4be930227848,5.0,REFRESH_STALE,Refresh content
3,4,content_2dfd17269502,5.0,REFRESH_STALE,Refresh content
4,5,content_fe5d259e6bc5,5.0,REFRESH_STALE,Refresh content
5,6,content_cd850fb019b1,5.0,REFRESH_STALE,Refresh content
6,7,content_a00c249b224d,5.0,REFRESH_STALE,Refresh content
7,8,content_fd6261b74d30,5.0,REFRESH_STALE,Refresh content
8,9,content_d87a116e2c79,5.0,REFRESH_STALE,Refresh content
9,10,content_4595e8704e07,5.0,REFRESH_STALE,Refresh content


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.